### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [27]:
%pip install numpy scikit-learn

/Users/bamazon/Documents/pln_1/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [28]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [29]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [30]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [31]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [32]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [33]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [34]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [35]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [36]:
# tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [37]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [38]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [39]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [40]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [41]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [42]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [43]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  1534, 10055,  4750], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [44]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [45]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [46]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [47]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](20,)","[480.,584.,591.,...,564.,465.,377.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](20,)","[-3.16,-2.96,-2.95,...,-3. ,-3.19,-3.4 ]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](20,)","[ 0, 1, 2,...,17,18,19]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](20, 101631)","[[0. ,0.94,0. ,...,0. ,0. ,0. ], [1.39,0.6 ,0. ,...,0. ,0. ,0. ], [0.95,0.14,0. ,...,0. ,0. ,0. ], ..., [0.42,2.9 ,0.04,...,0. ,0. ,0. ], [0.61,1.36,0. ,...,0. ,0. ,0. ], [0.03,0.38,0. ,...,0. ,0. ,0. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](20, 101631)","[[-11.56,-10.9 ,-11.56,...,-11.56,-11.56,-11.56], [-10.69,-11.1 ,-11.56,...,-11.56,-11.56,-11.56], [-10.9 ,-11.44,-11.57,...,-11.57,-11.57,-11.57], ..., [-11.22,-10.21,-11.54,...,-11.57,-11.57,-11.57], [-11.09,-10.7 ,-11.56,...,-11.56,-11.56,-11.56], [-11.53,-11.23,-11.56,...,-11.56,-11.56,-11.56]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,101631


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [48]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [49]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


# Consigna 1

In [55]:
rng = np.random.default_rng(42)
idx_random = rng.choice(X_train.shape[0], size=5, replace=False)

for idx in idx_random:
    cossim = cosine_similarity(X_train[idx], X_train)[0]
    mostsim = np.argsort(cossim)[::-1][1:6]

    print(f'=== DOC {idx} | clase: {newsgroups_train.target_names[y_train[idx]]} ===')
    print(newsgroups_train.data[idx][:400])
    print('--- 5 más similares ---')
    for i in mostsim:
        print(f'  doc {i} | sim={cossim[i]:.3f} | clase: {newsgroups_train.target_names[y_train[i]]}')
    print()

=== DOC 8754 | clase: talk.religion.misc ===

/(hudson)
/If someone inflicts pain on themselves, whether they enjoy it or not, they
/are hurting themselves.  They may be permanently damaging their body.

That is true.  It is also none of your business.  

Some people may also reason that by reading the bible and being a Xtian
you are permanently damaging your brain.  By your logic, it would be OK
for them to come into your home, take away yo
--- 5 más similares ---
  doc 6552 | sim=0.490 | clase: talk.religion.misc
  doc 10613 | sim=0.481 | clase: talk.religion.misc
  doc 3616 | sim=0.465 | clase: talk.religion.misc
  doc 8726 | sim=0.460 | clase: talk.politics.mideast
  doc 3902 | sim=0.459 | clase: talk.religion.misc

=== DOC 4965 | clase: comp.sys.mac.hardware ===

No.  Plug the printer in the printer port, and the modem in the modem
port. ;)
--- 5 más similares ---
  doc 5830 | sim=0.365 | clase: comp.sys.mac.hardware
  doc 9736 | sim=0.361 | clase: comp.sys.mac.hardware
  doc 1822

## Análisis - Consigna 1

Se seleccionaron 5 documentos de train al azar (con semilla fija para reproducibilidad) y se midió su similitud coseno contra los 11.314 documentos del conjunto, tomando los 5 más similares de cada uno y excluyendo el propio documento, que siempre encabeza el ranking con similitud 1.0. Además de comparar las etiquetas, se revisó el texto de cada vecino para entender qué términos producían la similitud.

### Coincidencia fuerte: DOC 8754 (`talk.religion.misc`)

Presentó las similitudes más altas de toda la muestra, entre 0.459 y 0.490, con 4 de 5 vecinos de la misma clase. El vocabulario del documento es marcadamente temático —*bible*, *Xtian*, *damaging*, *reason*— y son términos de IDF elevado que aparecen concentrados en el grupo religioso.

Al abrir el DOC 3902 (0.459) se observa que el contenido no es tan próximo como sugiere el número: el parentesco se sostiene en la aparición de términos religiosos generales (religión, Jesús, hábitos) más que en un solapamiento real de argumento. Es decir, la similitud alta refleja vocabulario compartido, no equivalencia de contenido.

### Clases hermanas: DOC 4965 y DOC 7404

En el DOC 4965 (`comp.sys.mac.hardware`) los vecinos se reparten entre `mac.hardware`, `ibm.pc.hardware` y `comp.graphics`, con similitudes entre 0.341 y 0.365. En el DOC 7404 (`comp.os.ms-windows.misc`) tres de los cinco vecinos son de `comp.windows.x`.

Esto no constituye un error del método: términos como *plug*, *port*, *printer*, *window*, *application* o *manager* pertenecen al vocabulario común de varios grupos de informática, y la coincidencia léxica es genuina aunque los grupos de noticias sean distintos. La diferencia entre un posteo sobre Mac y uno sobre PC es contextual, no léxica, y por lo tanto queda fuera del alcance de TF-IDF.

A esto se suma el efecto de la longitud. El DOC 4965 tiene solo dos líneas, por lo que su vector contiene muy pocos términos no nulos y la similitud queda determinada casi por completo por *printer* y *port*. Con vectores tan dispersos, una sola coincidencia desplaza el coseno de forma desproporcionada y el ranking se vuelve inestable.

### Ruido léxico: DOC 8719 como vecino del 7404

El DOC 8719 (`sci.med`) aparece con similitud 0.225 sin ninguna relación temática. Al revisar el texto se observa que la coincidencia proviene de la estructura del mensaje —fórmulas de saludo y cierre propias de un correo, y frases de cortesía como *please tell me how*— y no del asunto tratado. Este tipo de emparejamiento es esperable en un corpus donde, tras eliminar encabezados, firmas y citas, algunos documentos conservan poco contenido temático.

### El valor del coseno no predice el acierto: DOC 1009 vs DOC 7404

El DOC 1009 (`talk.politics.guns`) registra las similitudes más bajas de la muestra, entre 0.139 y 0.147 y separadas entre sí por milésimas, y sin embargo acierta 4 de 5 vecinos. El DOC 7404, con similitudes más altas (0.222 a 0.268), no acierta ninguno.

Esto muestra que el valor absoluto del coseno no anticipa el éxito de la clasificación: lo determinante es si los términos compartidos son específicos del tema. En el 1009, palabras como *firearms* o la referencia explícita al grupo *t.p.guns* son altamente discriminativas aunque su peso relativo sea bajo. El margen mínimo entre los cinco candidatos, en cambio, sí indica que el orden interno del ranking no es informativo: con diferencias de 0.008 la posición es prácticamente arbitraria.

### Conclusión

La similitud coseno sobre TF-IDF captura coincidencia **léxica**, no significado: el método no dispone de información semántica y no puede reconocer que dos términos distintos refieran a lo mismo. Funciona bien cuando el tema posee jerga propia y de baja frecuencia en el resto del corpus, y se degrada cuando los grupos comparten vocabulario técnico o cuando los documentos son demasiado breves para que el vector sea representativo.

# Consigna 2

In [66]:
sim_matrix = cosine_similarity(X_test, X_train)

idx_vecino = np.argmax(sim_matrix, axis=1)

y_pred_proto = y_train[idx_vecino]

print(f1_score(y_test, y_pred_proto, average='macro'))

0.5049911553681621


## Análisis — Consigna 2

Se implementó un clasificador por prototipos (1-Nearest Neighbor): cada documento de test se comparó por similitud coseno contra los 11.314 documentos de train y se le asignó la clase del más similar. El método no tiene entrenamiento no se ajustan parámetros ni existe función de pérdida, por lo que toda su capacidad reside en la representación TF-IDF y en la métrica de similitud.

El resultado fue un **F1 macro de 0.505**, por debajo del MultinomialNB base del notebook (~0.58) aunque muy por encima del azar, que en un problema de 20 clases rondaría 0.05.

La diferencia con Naïve Bayes se explica por la cantidad de evidencia que usa cada modelo. NB estima, a partir de los 11.314 documentos, qué términos son característicos de cada clase; una coincidencia léxica accidental queda diluida entre cientos de términos. El 1-NN, en cambio, apuesta toda la predicción a un único documento, si ese vecino se parece por ruido y no por tema, la clasificación se pierde por completo. El DOC 8719 de la consigna anterior es el ejemplo exacto emparentado con un documento de Windows por fórmulas de saludo propias de un correo, y un vecino así como más similar arrastra la predicción a una clase equivocada.

Dos factores observados en la consigna 1 agravan el problema. Los documentos breves, como el DOC 4965 de dos líneas, producen vectores con muy pocos términos no nulos, donde el vecino más cercano queda determinado por una o dos palabras y el ranking se vuelve inestable. Y las clases hermanas —`comp.sys.mac.hardware` frente a `comp.sys.ibm.pc.hardware`, o `comp.os.ms-windows.misc` frente a `comp.windows.x`— comparten vocabulario técnico: con un solo vecino, caer en la hermana equivocada es un error total, mientras que NB puede desempatar con diferencias sutiles de frecuencia entre clases.

Cabe señalar además el costo computacional. Aunque el método no requiere entrenamiento, la predicción exige comparar cada documento de test contra todo el conjunto de train, lo que produjo una matriz de 7.532 × 11.314 similitudes. Es el comportamiento inverso al de Naïve Bayes, que entrena una vez y luego predice de forma inmediata, y explica por qué este enfoque no escala.

## Consigna 3

In [71]:
tfidf_v2 = TfidfVectorizer(stop_words='english')

X_train_v2 = tfidf_v2.fit_transform(newsgroups_train.data)
X_test_v2 = tfidf_v2.transform(newsgroups_test.data)

for modelo in [MultinomialNB(alpha=0.01), ComplementNB(alpha=0.5)]:
    modelo.fit(X_train_v2, y_train)
    pred = modelo.predict(X_test_v2)
    print(type(modelo).__name__, f1_score(y_test, pred, average='macro'))

MultinomialNB 0.6844389919212164
ComplementNB 0.6978053768076979


In [72]:
configs = {
    'stop_words':            dict(stop_words='english'),
    'stop_words+sublinear':  dict(stop_words='english', sublinear_tf=True),
    'stop_words+min_df=2':   dict(stop_words='english', min_df=2),
    'stop_words+max_df=0.7': dict(stop_words='english', max_df=0.7),
    'sin stop_words':        dict(),
}

for nombre, params in configs.items():
    v = TfidfVectorizer(**params)
    Xtr = v.fit_transform(newsgroups_train.data)
    Xte = v.transform(newsgroups_test.data)
    for Modelo, a in [(MultinomialNB, 0.01), (ComplementNB, 0.5)]:
        m = Modelo(alpha=a).fit(Xtr, y_train)
        f1 = f1_score(y_test, m.predict(Xte), average='macro')
        print(f'{nombre:24} {Modelo.__name__:15} {f1:.4f}')

stop_words               MultinomialNB   0.6844
stop_words               ComplementNB    0.6978
stop_words+sublinear     MultinomialNB   0.6794
stop_words+sublinear     ComplementNB    0.6967
stop_words+min_df=2      MultinomialNB   0.6801
stop_words+min_df=2      ComplementNB    0.6974
stop_words+max_df=0.7    MultinomialNB   0.6844
stop_words+max_df=0.7    ComplementNB    0.6978
sin stop_words           MultinomialNB   0.6829
sin stop_words           ComplementNB    0.6961


## Análisis — Consigna 3

Se entrenaron modelos Naïve Bayes buscando maximizar el F1 macro en test, variando tanto el suavizado (`alpha`) como los parámetros de instanciación del `TfidfVectorizer`. En todos los casos se respetó la restricción de no modificar `ngram_range`.

### Barrido de alpha

Se exploró `alpha` en un rango de tres órdenes de magnitud (0.001 a 1.0) sobre el vectorizador con `stop_words='english'`. Ambos modelos presentan un óptimo interior, pero en posiciones muy distintas:

| modelo | mejor alpha | F1 macro |
|---|---|---|
| MultinomialNB | 0.01 | 0.6844 |
| ComplementNB | 0.5 | **0.6978** |

MultinomialNB alcanza su máximo con un suavizado cincuenta veces menor que ComplementNB, y decae hacia ambos lados (0.6689 en alpha=0.001, 0.6468 en alpha=1.0). ComplementNB recorre un rango de 0.6425 a 0.6978 en el mismo barrido, por lo que también resulta sensible al parámetro.

Esta diferencia es coherente con el mecanismo de cada modelo. MultinomialNB estima P(palabra | clase) con los documentos de cada clase —del orden de 400 en este corpus—, por lo que sus conteos son pequeños y un suavizado alto los distorsiona. ComplementNB estima con el complemento de cada clase, es decir cerca de 10.900 documentos, y sus conteos mayores admiten y requieren un suavizado proporcionalmente más grande.

### Comparación entre modelos

ComplementNB supera a MultinomialNB por 0.013 con ambos modelos optimizados. La ventaja se explica por la métrica de evaluación: el F1 macro pondera las 20 clases por igual, y ComplementNB fue diseñado precisamente para estabilizar las estimaciones de las clases minoritarias, que en MultinomialNB quedan mal representadas frente a las clases con más documentos.

Conviene señalar una precaución metodológica: comparados ambos en `alpha=0.7`, la diferencia parecía de 0.044, más del triple de la real. Evaluar modelos con hiperparámetros ajustados para uno solo exagera la brecha entre ellos.

### Parámetros del vectorizador

Se probaron cuatro configuraciones adicionales, manteniendo los alphas óptimos:

| configuración | MultinomialNB | ComplementNB |
|---|---|---|
| `stop_words='english'` | 0.6844 | 0.6978 |
| `+ sublinear_tf=True` | 0.6794 | 0.6967 |
| `+ min_df=2` | 0.6801 | 0.6974 |
| `+ max_df=0.7` | 0.6844 | 0.6978 |
| sin `stop_words` | 0.6829 | 0.6961 |

Ninguna variante mejoró la configuración base. Los resultados merecen tres observaciones.

`max_df=0.7` produce valores idénticos hasta el cuarto decimal, lo que indica que el filtro no descarta ningún término: tras eliminar las stopwords inglesas no queda vocabulario presente en más del 70% de los documentos, de modo que el vectorizador resultante es el mismo.

La eliminación de stopwords aporta apenas 0.0015. Esto es consistente con el funcionamiento de TF-IDF: el factor IDF ya asigna peso cercano a cero a los términos presentes en la mayoría de los documentos, por lo que filtrarlos explícitamente resulta en buena medida redundante.

`min_df=2` y `sublinear_tf` degradan levemente el desempeño (entre 0.004 y 0.005). En el caso de `min_df`, descartar los términos que aparecen en un solo documento elimina también vocabulario altamente discriminativo —jerga, nombres propios y referencias a grupos específicos—, del mismo tipo que resultó determinante en el análisis de la Consigna 1.

### Conclusión

El mejor resultado fue **ComplementNB con `alpha=0.5` y `stop_words='english'`, con F1 macro de 0.6978**, frente a ~0.58 del MultinomialNB base y 0.505 del clasificador por prototipos. La mejora provino casi enteramente del ajuste del suavizado y de la elección del modelo, mientras que los parámetros de filtrado del vocabulario resultaron irrelevantes en este corpus.

Debe advertirse que la optimización se realizó midiendo directamente sobre el conjunto de test, tal como plantea la consigna. Esto implica que 0.6978 es una estimación optimista del desempeño esperable sobre datos nuevos; una evaluación rigurosa requeriría un conjunto de validación independiente para la selección de hiperparámetros.

## Consigna 4

In [74]:
X_term = X_train.T

palabras = ["man", "woman", "peace", "power", "rules"]

for palabra in palabras:
    fila = tfidfvect.vocabulary_[palabra]
    
    cossim = cosine_similarity(X_term[fila], X_term)[0]
    mostsim = np.argsort(cossim)[::-1][1:6]
    
    print(f'=== {palabra} ===')
    for i in mostsim:
        print(f'  {idx2word[i]:20} sim={cossim[i]:.3f}')
    print()

=== man ===
  shintoism            sim=0.228
  prupose              sim=0.228
  confucianism         sim=0.228
  zoerasterism         sim=0.228
  mammal               sim=0.228

=== woman ===
  mdbs                 sim=0.423
  chauvinist           sim=0.423
  tithe                sim=0.406
  outraged             sim=0.369
  miriam               sim=0.365

=== peace ===
  withdraw             sim=0.307
  hamid                sim=0.257
  khan0095             sim=0.235
  esu                  sim=0.235
  eesau                sim=0.235

=== power ===
  supply               sim=0.303
  the                  sim=0.182
  pocatello            sim=0.179
  bartmich             sim=0.179
  8039                 sim=0.179

=== rules ===
  omnipotence          sim=0.218
  dumb                 sim=0.181
  boggled              sim=0.176
  niepornt             sim=0.176
  ttacs1               sim=0.176



In [77]:
X_term = X_train.T

palabras = ["water", "liquid", "ocean", "river", "lake"]

for palabra in palabras:
    fila = tfidfvect.vocabulary_[palabra]
    
    cossim = cosine_similarity(X_term[fila], X_term)[0]
    mostsim = np.argsort(cossim)[::-1][1:6]
    
    print(f'=== {palabra} ===')
    for i in mostsim:
        print(f'  {idx2word[i]:20} sim={cossim[i]:.3f}')
    print()

=== water ===
  towers               sim=0.397
  waynesboro           sim=0.389
  yorktown             sim=0.389
  dpw                  sim=0.389
  medford              sim=0.389

=== liquid ===
  supercooled          sim=0.537
  dense                sim=0.418
  comparision          sim=0.415
  creationists         sim=0.366
  tongs                sim=0.361

=== ocean ===
  shining              sim=0.727
  sunlight             sim=0.615
  mozart               sim=0.267
  venerable            sim=0.233
  nautilus             sim=0.200

=== river ===
  hasbani              sim=0.735
  litani               sim=0.680
  jordan               sim=0.463
  purify               sim=0.428
  flows                sim=0.402

=== lake ===
  finals               sim=0.384
  aquaintances         sim=0.382
  mutual               sim=0.291
  salt                 sim=0.272
  tahoe                sim=0.262



In [78]:
frecuencia = (X_train > 0).sum(axis=0).A1

for palabra in ['man', 'shintoism', 'river', 'hasbani', 'power', 'supply']:
    i = tfidfvect.vocabulary_[palabra]
    print(f'{palabra:15} aparece en {frecuencia[i]:5} documentos')

man             aparece en   407 documentos
shintoism       aparece en     2 documentos
river           aparece en    32 documentos
hasbani         aparece en     3 documentos
power           aparece en   589 documentos
supply          aparece en   157 documentos


## Análisis — Consigna 4

Se transpuso la matriz documento-término (`X_train.T`) para obtener la matriz término-documento, donde cada fila representa una palabra y sus columnas indican en qué documentos aparece y con qué peso. Aplicar similitud coseno sobre estas filas mide co-ocurrencia documental: dos palabras resultan similares si tienden a aparecer en los mismos documentos. Es la hipótesis distribucional en su forma más directa, sin ventana de contexto ni reducción de dimensionalidad.

Se seleccionaron manualmente diez términos en dos tandas: un grupo de vocabulario general y abstracto (`man`, `woman`, `peace`, `power`, `rules`) y otro de un campo semántico acotado (`water`, `liquid`, `ocean`, `river`, `lake`).

### Términos con vectores casi proporcionales

El resultado más llamativo aparece en los empates exactos. La consulta de `man` devuelve cinco vecinos con similitud idéntica de 0.228 —*shintoism*, *prupose*, *confucianism*, *zoerasterism*, *mammal*— y el mismo fenómeno se repite en `peace`, `power`, `rules`, `water` y `woman`.

Al consultar directamente `shintoism` se obtiene la confirmación del mecanismo: *zoerasterism* y *prupose* alcanzan similitud 1.000, y la propia palabra consultada reaparece en el ranking con ese mismo valor. Esto ocurre porque existen varios términos con vectores exactamente idénticos, empatados en el máximo, de modo que el descarte de la primera posición resulta insuficiente para excluir la palabra original.

El conteo de frecuencias explica el fenómeno: *shintoism* aparece en 2 documentos y *hasbani* en 3, frente a los 407 de `man` o los 589 de `power`. Cuando dos términos aparecen en los mismos dos o tres documentos y en ningún otro, sus vectores son prácticamente proporcionales y la similitud coseno se aproxima a 1 sin que exista relación semántica alguna. El factor IDF acentúa el efecto: un término presente en 2 de 11.314 documentos recibe un peso muy elevado, de manera que basta compartir un documento para que el coseno se dispare.

El tipo de tokens involucrados confirma el diagnóstico: errores de tipeo (*prupose*, *comparision*, *aquaintances*), identificadores de usuario (*khan0095*, *ttacs1*, *bartmich*), números (*8039*, *230watt*) y topónimos aislados (*waynesboro*, *pocatello*). La métrica favorece estructuralmente a los términos más raros del corpus, que son también los menos informativos.

Vale notar la tensión con el resultado de la Consigna 3, donde `min_df=2` degradó el F1. Ambos hallazgos son compatibles: los términos de muy baja frecuencia son mayoritariamente ruido, pero una fracción de ellos —jerga y nombres propios del dominio— resulta altamente discriminativa para la clasificación.

### Casos de recuperación semántica efectiva

El mejor resultado corresponde a `river`, cuyos vecinos son *hasbani* (0.735), *litani* (0.680), *jordan* (0.463), *purify* (0.428) y *flows* (0.402). El Hasbani y el Litani son ríos del sur del Líbano, y el Jordán es central en la disputa regional por los recursos hídricos. La consulta recíproca de `hasbani` devuelve *litani* (0.855), *river* (0.735) y los términos hidrológicos *rainfall*, *inflow* y *climatic* (0.698 los tres).

El campo semántico recuperado es coherente, pero no es el de geografía general sino el de `talk.politics.mideast`. Esto ilustra con precisión el alcance del método: la co-ocurrencia documental no devuelve el significado de diccionario de una palabra, sino el contexto temático dominante en el que el corpus la utiliza.

Otros casos exitosos incluyen `power` → *supply* (0.303) y `lake` → *salt*, *tahoe*, donde el método recupera colocaciones reales —"power supply", "Salt Lake", "Lake Tahoe"— pese a operar únicamente con unigramas. La relación entre `power` y `supply` se sostiene en ambas direcciones de consulta, lo que indica una co-ocurrencia genuina y no un artefacto. Resulta notable que incluso el ruido que acompaña a `supply` (*230watt*, *4x1mb*, *83d87*) proviene del dominio correcto: publicaciones de venta de componentes de hardware.

### La concentración temática pesa más que la frecuencia

El contraste entre términos revela un resultado contraintuitivo:

| término | documentos | calidad de los vecinos |
|---|---|---|
| `power` | 589 | una colocación válida, resto ruido |
| `man` | 407 | ruido completo |
| `supply` | 157 | colocación válida, ruido del dominio |
| `river` | 32 | campo semántico coherente |
| `hasbani` | 3 | campo semántico coherente |
| `shintoism` | 2 | vectores idénticos, sin relación |

Las palabras más frecuentes produjeron los peores resultados. `man`, presente en 407 documentos, devolvió exclusivamente ruido, mientras que `river`, con apenas 32, devolvió el campo semántico más consistente de todo el ejercicio.

La explicación es que la similitud coseno mide el perfil de co-ocurrencia y no la cantidad de apariciones. Un término distribuido entre 407 documentos repartidos por los 20 grupos temáticos genera un vector difuso, sin dirección definida: ninguna otra palabra acompaña ese patrón, de modo que el ranking termina dominado por términos raros que comparten algún documento de peso elevado. `river`, en cambio, se concentra en un conjunto reducido de documentos sobre un mismo tema, produce un vector de dirección nítida y permite que se le alineen otras palabras de ese contexto.

Los dos extremos del cuadro fallan por motivos opuestos: `shintoism` por concentración excesiva —tan pocos documentos que cualquier coincidencia produce proporcionalidad—, y `man` por dispersión —tantos contextos distintos que ninguno define su dirección.

### Conclusión

La similitud coseno sobre la matriz término-documento recupera relaciones temáticas y colocaciones, pero no sinonimia ni significado. Su desempeño no depende de la frecuencia del término sino de su concentración temática: funciona con vocabulario específico de un dominio acotado y falla tanto con términos polisémicos y transversales como con aquellos de frecuencia muy baja.

Estas limitaciones son las que motivan el desarrollo de representaciones densas. Un modelo de embeddings, al trabajar con ventanas de contexto locales en lugar de documentos completos y al reducir la dimensionalidad, permite que términos con distribución dispersa adquieran una representación estable y que palabras semánticamente equivalentes queden próximas en el espacio, algo que la co-ocurrencia documental cruda no puede lograr.